In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# Overnight run: encoder saturation, India label efficiency, and RSNA -> India
# few-shot transfer
#
# Cells 1-7 below are nb2's, verbatim, except for one addition to `Reducer`:
# `set_encoder_norm()`, a non-affine BatchNorm between the learned projection and
# the tanh. It adds no parameters and is a no-op on the PCA reducers.
#
# Everything is resumable. If the runtime dies or the budget runs out, re-run the
# notebook and it continues from the log on Drive.
# 1. Drive
import os

try:
    from google.colab import drive
    drive.mount(P["drive_mount"])
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not in Colab -- point OUTPUT_DIR at wherever Phase A wrote its caches.")
if IN_COLAB and not os.path.isdir(P["drive_root"]):
    raise RuntimeError("Drive did not finish mounting -- re-run this cell.")
print("ok")

In [ ]:
# 2. Installs
#
# Phase B needs far less than Phase A: no MONAI, no pydicom, no SimpleITK. Only PennyLane
# (the circuits), torch (the graph they live in), and pyarrow (to read the feature tables).
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pennylane>=0.35,<0.50", "pyarrow>=14", "pandas>=2.1", "scikit-learn>=1.3"],
               check=False)

import pennylane as qml
import torch

print(f"pennylane {qml.__version__}   torch {torch.__version__}")
print("NOTE: this notebook does not need a GPU. A CPU runtime is the right choice --")
print("      it leaves your GPU quota for Phase A, where it actually matters.")

In [ ]:
# 3. CONFIGURATION
import os

# Must match Phase A.
OUTPUT_DIR   = P["rsna_output_dir"]
FEATURE_DIR  = os.path.join(OUTPUT_DIR, "features")
SPATIAL_DIR  = os.path.join(OUTPUT_DIR, "spatial_maps")

# ---- the grid -------------------------------------------------------------------------
# Comment out any backbone you have not extracted yet; the notebook reports what it found
# and runs the rest rather than failing.
BACKBONES_TO_RUN = [
    "B1_swinunetr_suprem",
    "B2_segresnet_suprem",
    "B3_resnet18_medicalnet",
    # "B4_swinunetr_ssl5050",
]
# Q3_quanv is REMOVED here on purpose, and it is not an oversight.
#
# Every arm this notebook fits is a POOLED one -- E0 audits the pooled families, and the E6
# and E7 curves use CURVE_FAMILY, which is pooled too. Leaving Q3_quanv in this list sets
# NEED_SPATIAL in the next cell, which then pulls roughly 11,600 individual .npy files off
# Drive (about 500 MB, and Drive is latency-bound on small files) to build maps that nothing
# below ever reads. On a Colab runtime that is easily half an hour of the night spent on
# nothing. Put it back if you add a spatial arm to the curves.
# E0 now measures the POOLED and the SPATIAL arm in the same harness, on the same splits and
# the same seeds, because that comparison is the paper's central claim and the two numbers
# previously came from different holdouts. That needs the cached spatial maps, so Q3_quanv is
# back in this list. It costs a one-off Drive read of ~11,600 small files (~500 MB) at the
# start of the run. Set E0_INCLUDE_SPATIAL = False below to skip that.
E0_INCLUDE_SPATIAL = True
FAMILIES_TO_RUN = (["Q1_sel", "Q2_reupload", "Q4_ttn"]
                   + (["Q3_quanv"] if E0_INCLUDE_SPATIAL else []))

# ---- the reducer ----------------------------------------------------------------------
# How a 768-2304-d feature vector becomes N_QUBITS rotation angles matters MORE than the
# choice of ansatz. Running one and presenting it as neutral is the easiest way to mislead
# yourself, so the grid runs on the primary and the finalists are re-run on all three.
#
#   "pca_supervised"  PCA on the training fold, then the components most correlated with
#                     the label. Nothing outside the adapter is trainable.
#   "trainable"       a trainable Linear(d -> n_qubits) trained jointly with the circuit.
#                     Highest performing; holds most of the parameters, which is exactly
#                     why the classical control shares it and only the TAIL is matched.
#   "pca"             plain top-variance PCA. Label-blind, and the honest floor.
PRIMARY_REDUCER  = "pca_supervised"
FINALIST_REDUCERS = ["pca_supervised", "trainable", "pca"]

# ---- targets --------------------------------------------------------------------------
PRIMARY_TARGET   = "any"          # ICH presence: the powered endpoint
SECONDARY_TARGETS = ["intraparenchymal", "subdural", "subarachnoid", "intraventricular",
                     "epidural"]
RUN_SECONDARY_ON_FINALIST_ONLY = True    # subtypes are exploratory; see the outline
MIN_POSITIVES    = 30

# ---- circuit --------------------------------------------------------------------------
N_QUBITS   = 4       # must be a power of two: Q4's tree ansatz needs one
DEPTH      = 3       # SEL layers (Q1/Q3) and re-uploads (Q2); Q4's depth is fixed by the tree
DIFF_METHOD = "backprop"      # measured against adjoint in the benchmark cell below
INIT_STD   = 0.01             # near-identity init: barren-plateau mitigation

# ---- optimisation ---------------------------------------------------------------------
EPOCHS          = 80
EPOCHS_SPATIAL  = 25      # Q3 sees n_positions times more circuit evaluations per epoch
BATCH           = 128
BATCH_SPATIAL   = 32
LR              = 0.05
ANGLE_L2_BETA   = 1e-4    # penalty on the ROTATION ANGLES specifically
PROJ_WEIGHT_DECAY = 1e-3  # on the reducer/projection only
FOCAL_ALPHA, FOCAL_GAMMA = 0.25, 2.0

# ---- protocol ---------------------------------------------------------------------------
CV_FOLDS      = 5
HOLDOUT_FRAC  = 0.2
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP   = 2000
SEED          = 12345
NISQ_P        = 0.01
FORCE_RERUN   = False

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "phaseb_results.jsonl")
RESULTS_CSV   = os.path.join(OUTPUT_DIR, "phaseb_leaderboard.csv")
FIG_DIR       = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

if N_QUBITS & (N_QUBITS - 1):
    raise SystemExit(f"N_QUBITS must be a power of two (Q4's tree ansatz halves the wires "
                     f"at every level); got {N_QUBITS}")

CFG = dict(n_qubits=N_QUBITS, depth=DEPTH, diff_method=DIFF_METHOD, init_std=INIT_STD,
           epochs=EPOCHS, epochs_spatial=EPOCHS_SPATIAL, batch=BATCH,
           batch_spatial=BATCH_SPATIAL, lr=LR, angle_l2_beta=ANGLE_L2_BETA,
           projection_weight_decay=PROJ_WEIGHT_DECAY, focal_alpha=FOCAL_ALPHA,
           focal_gamma=FOCAL_GAMMA, cv_folds=CV_FOLDS, holdout_frac=HOLDOUT_FRAC,
           seed=SEED, n_bootstrap=N_BOOTSTRAP, target_sensitivity=TARGET_SENSITIVITY,
           predict_batch=256, dataset="rsna-ich")
CFG["signature"] = "|".join([
    f"q={N_QUBITS}", f"d={DEPTH}", f"diff={DIFF_METHOD}", f"ep={EPOCHS}/{EPOCHS_SPATIAL}",
    f"lr={LR}", f"bs={BATCH}/{BATCH_SPATIAL}", f"beta={ANGLE_L2_BETA}",
    f"wd={PROJ_WEIGHT_DECAY}", f"focal=({FOCAL_ALPHA},{FOCAL_GAMMA})",
    f"folds={CV_FOLDS}", f"holdout={HOLDOUT_FRAC}", f"seed={SEED}",
])
print("config signature:", CFG["signature"])
print(f"grid: {len(BACKBONES_TO_RUN)} backbones x {len(FAMILIES_TO_RUN)} families "
      f"= {len(BACKBONES_TO_RUN)*len(FAMILIES_TO_RUN)} cells")

In [ ]:
# 4. Load every backbone's cached features, and the spatial maps the Q3 arm needs
#
# The three backbones do NOT share a feature width (768/128/512 per window x 3 windows),
# which is fine -- each is its own row of the grid and the reducer absorbs the difference.
# What they MUST share is the set of series, because otherwise a "difference between
# backbones" is partly a difference between cohorts. The intersection is taken explicitly
# and reported, rather than each backbone being scored on whatever it happens to have.
import concurrent.futures as _cf
import time as _time

import numpy as np
import pandas as pd

SPATIAL_LOAD_WORKERS = 16     # Drive is latency-bound; this is round trips, not bytes

LABEL_COLS = ["any", "epidural", "intraparenchymal", "intraventricular", "subarachnoid",
              "subdural"]

TABLES, MISSING = {}, []
for b in BACKBONES_TO_RUN:
    p = os.path.join(OUTPUT_DIR, f"features_{b}.parquet")
    if not os.path.exists(p):
        MISSING.append(b)
        continue
    TABLES[b] = pd.read_parquet(p)
    print(f"{b:<26} {len(TABLES[b]):>6,} series x "
          f"{sum(c.startswith('f_') for c in TABLES[b].columns):>5} features")
if MISSING:
    print(f"\nNOT FOUND (skipped): {MISSING}")
    print("  run nb1_phaseA_extract.ipynb with BACKBONE_ID set to each of these first.")
if not TABLES:
    raise SystemExit("no feature tables found -- Phase A has not produced anything yet.")

# The intersection is what protects this notebook from a real hazard in Phase A: cell 15
# there admits any series whose progress record says "ok" under the current preprocessing
# signature, WITHOUT checking that the series is in the current manifest. B1 was extracted
# once before on an earlier 86-series cohort under an identical signature, so 68 of those
# came along into features_B1_swinunetr_suprem.parquet. Because Phase A merges metadata with
# a LEFT join, those rows carry NaN for every label and for `patient` -- and a NaN in
# `patient` would silently corrupt the patient-grouped split that the whole protocol rests
# on. Taking the intersection FIRST removes them; the assertions below prove it.
common = None
for b, t in TABLES.items():
    s = set(t["series"])
    common = s if common is None else (common & s)
common = sorted(common)
print(f"\nseries present in ALL {len(TABLES)} backbone(s): {len(common):,}")
for b, t in TABLES.items():
    if len(t) != len(common):
        _extra = sorted(set(t["series"]) - set(common))
        _nan = int(t[t["series"].isin(_extra)]["any"].isna().sum()) if "any" in t.columns else 0
        print(f"  {b}: dropping {len(t)-len(common):,} series not shared by every backbone"
              + (f"   ({_nan:,} of them have no label at all -- out-of-cohort leftovers)"
                 if _nan else ""))
if len(common) < 50:
    raise SystemExit(f"only {len(common)} shared series -- extract more before running the grid.")

X_BY_BACKBONE, M_BY_BACKBONE = {}, {}
meta = None
for b, t in TABLES.items():
    t = t[t["series"].isin(common)].sort_values("series").reset_index(drop=True)
    fcols = [c for c in t.columns if c.startswith("f_")]
    X_BY_BACKBONE[b] = t[fcols].to_numpy(dtype=np.float32)
    if meta is None:
        meta = t[["series", "patient", "study"] +
                 [c for c in LABEL_COLS if c in t.columns]].copy()

# Every downstream split is grouped by these. A NaN here does not raise -- it becomes the
# string "nan", silently collapsing unrelated patients into ONE group and leaking scans of
# different people across the train/test boundary. Checked rather than trusted.
if meta["patient"].isna().any():
    raise SystemExit(
        f"{int(meta['patient'].isna().sum()):,} of {len(meta):,} rows have no patient id. "
        f"They are series that are in a feature table but not in manifest.parquet, so the "
        f"metadata merge in Phase A left them empty. Every patient-grouped split would be "
        f"wrong. Fix the feature tables (restrict them to manifest.parquet) before running.")
for _lc in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if _lc in meta.columns and meta[_lc].isna().any():
        raise SystemExit(f"{int(meta[_lc].isna().sum()):,} rows have no {_lc!r} label; "
                         f"same cause and same fix as above.")

SERIES_ALL = meta["series"].to_numpy()
GROUPS_ALL = meta["patient"].astype(str).to_numpy()   # group by PATIENT, never by series
assert len(set(GROUPS_ALL)) > 1, "every series resolved to one patient group"
print(f"patients: {len(set(GROUPS_ALL)):,}  ({len(SERIES_ALL)/max(len(set(GROUPS_ALL)),1):.2f} "
      f"series per patient)")

# ---- spatial maps (Q3 only) -------------------------------------------------------------
NEED_SPATIAL = any(f == "Q3_quanv" for f in FAMILIES_TO_RUN)
if NEED_SPATIAL:
    for b in list(X_BY_BACKBONE):
        d = os.path.join(SPATIAL_DIR, b)
        if not os.path.isdir(d):
            print(f"\nNO SPATIAL MAPS for {b} at {d}")
            print("  Q3 is the deep-placement arm and cannot run from pooled vectors. Either")
            print("  re-run Phase A for this backbone with SAVE_SPATIAL_MAP=True, or drop")
            print("  'Q3_quanv' from FAMILIES_TO_RUN.")
            continue
        # SERIAL LOADING HERE IS THE SAME MISTAKE PHASE A MADE. This is 3,889 individual
        # .npy files per backbone, ~11,700 across three, and on a Drive FUSE mount it is the
        # per-file round trip that costs -- not the bytes. One at a time that is 20 minutes
        # to 3 hours before a single circuit trains; a thread pool makes it a couple of
        # minutes. Order is preserved by index, NOT by completion order: these rows are
        # aligned with meta["series"] and with X_BY_BACKBONE, so shuffling them would pair
        # every spatial map with the wrong label.
        _ids = list(meta["series"])
        _paths = [os.path.join(d, f"{s}.npy") for s in _ids]
        arrs = [None] * len(_paths)

        def _one(i, _p=_paths):
            try:
                return i, np.load(_p[i])
            except Exception:
                return i, None

        _t0 = _time.time()
        with _cf.ThreadPoolExecutor(max_workers=SPATIAL_LOAD_WORKERS) as _ex:
            for _n, (_i, _a) in enumerate(_ex.map(_one, range(len(_paths))), 1):
                arrs[_i] = _a
                if _n % 500 == 0 or _n == len(_paths):
                    print(f"\r  {b}: loaded {_n:,}/{len(_paths):,} maps "
                          f"({_time.time()-_t0:.0f}s)   ", end="", flush=True)
        print()
        _n_missing = sum(1 for _a in arrs if _a is None)
        if _n_missing:
            print(f"  {b}: {_n_missing:,} spatial map(s) missing on disk")
        ok = _n_missing == 0
        if ok and arrs:
            _shapes = {a.shape for a in arrs}
            if len(_shapes) != 1:
                print(f"  {b}: spatial maps have {len(_shapes)} different shapes {list(_shapes)[:3]}"
                      f" -- Q3 skipped for this backbone")
                ok = False
        if ok and arrs:
            M_BY_BACKBONE[b] = np.stack(arrs).astype(np.float32)
            print(f"spatial maps {b:<26} {M_BY_BACKBONE[b].shape}  "
                  f"({M_BY_BACKBONE[b].nbytes/1e6:.0f} MB, "
                  f"{int(np.prod(M_BY_BACKBONE[b].shape[2:]))} positions/series)")
        else:
            print(f"spatial maps {b}: incomplete -- Q3 will be skipped for this backbone")

# ---- targets ----------------------------------------------------------------------------
print(f"\n{'target':<22}{'positives':>11}{'prevalence':>13}{'majority acc':>15}")
TARGETS = {}
for t in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if t not in meta.columns:
        continue
    y = (meta[t].to_numpy() > 0).astype(int)
    npos = int(y.sum())
    flag = ""
    if npos < MIN_POSITIVES or (len(y) - npos) < MIN_POSITIVES:
        flag = f"  <-- below MIN_POSITIVES={MIN_POSITIVES}, skipped"
    else:
        TARGETS[t] = y
    print(f"  {t:<20}{npos:>11,}{y.mean():>13.4f}{max(y.mean(),1-y.mean()):>15.3f}{flag}")
if PRIMARY_TARGET not in TARGETS:
    raise SystemExit(f"the primary target {PRIMARY_TARGET!r} does not have enough positives.")
print("\n  'majority acc' is what you score by always answering the majority class. Any")
print("  accuracy at or below it is worth nothing, which is why every accuracy below is")
print("  printed next to it.")

In [ ]:
# 5. The four quantum-circuit families, the adapters, and the training loop
#
# Moved verbatim to src/qct3d/models.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.models import *  # noqa: F401,F403

In [ ]:
# 6. The protocol: patient-grouped splits, the three arms, and the paired comparison
#
# Moved verbatim to src/qct3d/protocol.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.protocol import *  # noqa: F401,F403

In [ ]:
# 7. Parameter budgets, and a self-test that proves gradients reach the rotation angles
#
# ASSERTED, because each failure means the cell is broken: the angles must MOVE during
# training (if they do not, no gradient is reaching the circuit and every quantum number
# below is an untrained random circuit's output), the loss must go DOWN, and the outputs
# must be finite.
#
# REPORTED, not asserted: the AUROC on a planted signal. How high it gets depends on the
# epoch count, the learning rate and the qubit count, so asserting a floor on it fires
# when nothing is wrong -- an alarm that goes off spuriously is worse than no alarm.
import numpy as np
import torch

print(f"{'family':<14}{'role':<46}{'tail':>6}{'hidden':>8}{'control':>9}{'err':>7}")
print("-" * 92)
BUDGETS = {}
for fam in FAMILIES_TO_RUN:
    probe = AdapterModel("quantum", Reducer("pca", 64, N_QUBITS), N_QUBITS, DEPTH,
                         family=fam, diff_method="backprop")
    tail = probe.tail_params()
    hid, err, cpar = match_hidden_width(N_QUBITS, tail)
    BUDGETS[fam] = {"tail": tail, "hidden": hid, "control_params": cpar, "err": err}
    role = QUANTUM_FAMILIES[fam]["role"]
    print(f"{fam:<14}{role[:44]:<46}{tail:>6}{hid:>8}{cpar:>9}{err:>6.1%}")
    if err > 0.10:
        print(f"  *** {fam}: outside the +/-10% matching band. With this few parameters the "
              f"hidden width is quantised too coarsely -- report the exact counts in the paper "
              f"rather than claiming a match. ***")
print(f"\nlinear probe floor: the full feature vector + 1 bias, no adapter, no reduction")


def _self_test(fam):
    rng = np.random.default_rng(0)
    n, d, k = 600, 128, 20
    L = rng.standard_normal((d, k)).astype("float32")
    Z = rng.standard_normal((n, k)).astype("float32")
    X = np.maximum(Z @ L.T + 0.5 * rng.standard_normal((n, d)), 0).astype("float32")
    s = Z @ rng.standard_normal(k) + rng.standard_normal(n)
    y = (s >= np.quantile(s, 0.86)).astype(int)
    tr, te = np.arange(450), np.arange(450, n)
    red = Reducer("trainable", d, N_QUBITS, seed=0).fit(X[tr], y[tr], seed=0)
    m = AdapterModel("quantum", red, N_QUBITS, DEPTH, family=fam,
                     diff_method=DIFF_METHOD, init_std=INIT_STD, seed=0)
    Ztr, Zte = red.transform_np(X[tr]), red.transform_np(X[te])
    before = m.q.weights.detach().clone()
    Xt = torch.as_tensor(Ztr, dtype=torch.float32)
    yt = torch.as_tensor(y[tr], dtype=torch.float32)
    with torch.no_grad():
        l0 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    train_adapter(m, Ztr, y[tr], epochs=40, lr=LR, batch=BATCH, beta=ANGLE_L2_BETA,
                  weight_decay=PROJ_WEIGHT_DECAY, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=0)
    with torch.no_grad():
        l1 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    moved = float((m.q.weights.detach() - before).abs().max())
    p = predict_proba(m, Zte)
    return moved, l0, l1, fast_auroc(y[te], p), bool(np.isfinite(p).all())


print("\nself-test on synthetic features with a planted signal (~14% prevalence):")
for fam in [f for f in FAMILIES_TO_RUN if not QUANTUM_FAMILIES[f]["spatial"]]:
    mv, l0, l1, auc, fin = _self_test(fam)
    print(f"  {fam:<14} angles moved {mv:.5f} | focal {l0:.5f} -> {l1:.5f} | "
          f"AUROC {auc:.3f} | finite {fin}")
    assert fin, f"{fam}: non-finite probabilities -- the circuit is broken."
    assert mv > 1e-6, (
        f"{fam}: the rotation angles did not move during training (max change {mv:.2e}). No "
        f"gradient is reaching the circuit, so every number this notebook produces for this "
        f"family would be an untrained random circuit's output. Check DIFF_METHOD.")
    assert l1 < l0, (f"{fam}: the focal loss did not decrease ({l0:.5f} -> {l1:.5f}). The "
                     f"optimiser is making no progress; check LR (currently {LR}).")
print("  -> gradients reach the rotation angles and the loss falls: the circuits train.")

In [ ]:
# ==============================================================================================
# OV-1. OVERNIGHT CONFIGURATION -- budgets, resume, and the one knob the whole paper turns on
# ==============================================================================================
# This notebook reuses nb2's cells 1-7 VERBATIM above (config, feature loading, the circuit
# families, the adapters, the training loop, the protocol, the budgets). Nothing about the
# models changes, so every number here is directly comparable with the Phase-B grid.
#
# What is added below, in priority order, is three experiments:
#
#   E0  SATURATION AUDIT + ENCODER-NORM PARITY.
#       Q1/Q2/Q4 with reducer mode "trainable" have a learned nn.Linear feeding tanh and NO
#       normalisation. Q3 has a learned conv feeding tanh WITH a non-affine BatchNorm, added
#       when Q3 was found stuck at AUROC exactly 0.500. The documented cause was weight growth
#       under AdamW driving pre-tanh magnitude to 30-45, where tanh is flat to machine
#       precision. That mechanism is input-independent, so the trainable pooled arms are
#       exposed to it too -- and the 9,261-parameter-matched comparison that the "placement
#       dominates ansatz" claim rests on is exactly Q1_trainable vs Q3. E0 measures pre-tanh
#       magnitude for every arm and re-runs the pooled arms with the same normalisation.
#
#       NOTE, and this matters for how you write it up: the PCA reducers cannot saturate.
#       They have no trainable projection and their components are divided by their own std
#       at fit time, so pre-tanh is O(1) by construction. Table 1's Q1 = 0.749 uses
#       pca_supervised and is NOT at risk. The exposed cell is Q1 with mode "trainable".
#
#   E6  INDIA LABEL-EFFICIENCY CURVE. Train from scratch on India only at k local labels.
#       This is the direct analogue, on real clinical data, of the MNIST experiment behind
#       the published claim that quantum adapters degrade more gracefully when labels are
#       scarce. normal_vs_abnormal carries 859 positives and is the powered target.
#
#   E7  RSNA -> INDIA FEW-SHOT TRANSFER. Train the adapter on RSNA, then adapt it on k India
#       labels. Zero-shot is the 0.490 cell already measured; this asks how many local labels
#       buy it back, and whether quantum buys it back faster than a parameter-matched
#       classical tail.
#
# Everything is RESUMABLE and BUDGETED. Each unit of work writes one JSON line as it
# finishes, keyed by everything that defines it, so re-running skips what is done. When
# WALL_BUDGET_MIN is spent the notebook stops cleanly between units and tells you what is
# left. Run it again tomorrow night to continue.
import json
import os
import time

import numpy as np

# ---- where India lives -------------------------------------------------------------------
INDIA_DIR = P["external_output_dir"]
INDIA_LABEL_SOURCE = "auto"      # "auto" finds matched_table.parquet in the usual folders

# ---- what to run, in order. Drop names to skip a stage. -----------------------------------
STAGES = ["E0", "E6", "E7"]

# ---- wall clock -------------------------------------------------------------------------
WALL_BUDGET_MIN = 420            # 7 h. The loop stops cleanly between units, never mid-fit.
OV_LOG = os.path.join(OUTPUT_DIR, "overnight_results.jsonl")

# ---- E0 -----------------------------------------------------------------------------------
E0_FAMILIES = ["Q1_sel", "Q2_reupload", "Q4_ttn"]
E0_REDUCERS = ["trainable"]      # the only reducer that CAN saturate
E0_NORMS    = [False, True]      # False reproduces the current grid; True is the parity run
E0_BACKBONES = None              # None = every backbone in BACKBONES_TO_RUN
E0_SEEDS    = [12345, 1, 2, 3, 4]
# FIVE SEEDS, because E0 is now the paper's central table and its central claim is a
# comparison. One seed gives point estimates and no way to say whether pooled-vs-spatial is a
# real difference or training noise -- which is exactly the criticism the whole project is
# open to. Five is cheap here: a pooled fit is 12 s and a spatial fit a few minutes.
E0_SPATIAL_FAMILY = "Q3_quanv" if globals().get("E0_INCLUDE_SPATIAL", False) else None
# The spatial arm goes through the SAME splits, seeds and scoring as the pooled arms. Its
# 0.867 in the Phase-B grid came from a different holdout, so the two were never directly
# comparable -- and "placement dominates ansatz" is a comparison between exactly those two
# numbers. This makes it one measurement.

# ---- E6 / E7 ------------------------------------------------------------------------------
# Budgets are in LABELLED SERIES, drawn patient-grouped. 0 means zero-shot (E7 only).
LABEL_BUDGETS   = [25, 50, 100, 200, 400, 900]
LABEL_BUDGETS_T = [0, 10, 25, 50, 100, 200, 400, 900]
N_DRAWS         = 50             # independent draws per budget; the curve is the result
#
# MEASURED COST, so you can size this yourself. On 2 CPU threads, one fit takes:
#   quantum   4.1 s at n=25,  12.5 s at n=900,  49 s at n=3,889
#   classical 0.1 s          1.0 s              4.0 s
#   linear    0.01 s         0.14 s             0.8 s
# The quantum arm dominates and the whole default run below is on the order of 3-5 hours
# for all three backbones at 50 draws. If you want it shorter, cut N_DRAWS or CURVE_BACKBONES
# first -- never the budgets, because the SHAPE of the curve is the result.
CURVE_BACKBONES = ["B1_swinunetr_suprem", "B2_segresnet_suprem", "B3_resnet18_medicalnet"]
# Three backbones is three independent replications of the same curve. That is worth far
# more to a reviewer than one curve with tighter error bars, and it is affordable here.
CURVE_FAMILY    = "Q1_sel"       # pooled arm: comparable to the published MNIST experiment
CURVE_REDUCER   = "trainable"
CURVE_ARMS      = ["quantum", "classical", "linear_probe"]
CURVE_TARGETS_INDIA = ["abnormal_any", "hemorrhage"]   # powered first, then the clinical one
ADAPT_MODES     = ["tail", "tail_plus_projection"]     # E7 only; "readout" is added free

ENCODER_NORM_DEFAULT = True      # what E6/E7 use, so their arms are never the saturated ones

os.makedirs(os.path.dirname(OV_LOG) or ".", exist_ok=True)
_T0 = time.time()


def budget_left_min():
    return WALL_BUDGET_MIN - (time.time() - _T0) / 60.0


def out_of_budget(need_min=2.0):
    return budget_left_min() < need_min


def ov_done_keys(path=None):
    """Every unit key already on the log. The log is append-only; last line wins."""
    path = path or OV_LOG
    done = {}
    if os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("key"):
                    done[r["key"]] = r
    return done


def ov_record(key, **fields):
    rec = {"key": key, "t": time.strftime("%Y-%m-%d %H:%M:%S"), **fields}
    with open(OV_LOG, "a") as fh:
        fh.write(json.dumps(rec, default=float) + "\n")
    return rec


OV_DONE = ov_done_keys()
print("=" * 94)
print("OVERNIGHT RUN")
print("=" * 94)
print(f"  stages          : {STAGES}")
print(f"  wall budget     : {WALL_BUDGET_MIN} min")
print(f"  results log     : {OV_LOG}")
print(f"  already done    : {len(OV_DONE):,} unit(s) -- these will be skipped")
print(f"  label budgets   : E6 {LABEL_BUDGETS}  |  E7 {LABEL_BUDGETS_T}")
print(f"  draws per point : {N_DRAWS}")
print(f"  curve backbones : {CURVE_BACKBONES}")
print("=" * 94)


In [ ]:
# ==============================================================================================
# OV-2. LOAD THE INDIA COHORT into the same structures nb2 already uses
# ==============================================================================================
# X_IND / M_IND mirror X_BY_BACKBONE / M_BY_BACKBONE, and Y_IND / G_IND mirror TARGETS /
# GROUPS_ALL. Everything downstream then treats the two cohorts identically, which is the
# only way the comparison means anything.
#
# The India feature tables were written by nb5 with columns emb_0000..., series_uid,
# patient_id, study_uid. nb2's RSNA tables use f_0000..., series, patient, study. They are
# renamed onto nb2's schema here rather than duplicating parquet.
import glob

import pandas as pd

X_IND, M_IND, Y_IND = {}, {}, {}
IND_META = None


def _india_label_table():
    """matched_table.parquet, wherever cell 7 of nb5 actually wrote it."""
    if INDIA_LABEL_SOURCE != "auto" and os.path.exists(INDIA_LABEL_SOURCE):
        return pd.read_parquet(INDIA_LABEL_SOURCE)
    cands = [os.path.join(d, "matched_table.parquet") for d in (
        INDIA_DIR,
        P["pillar0_external_output_dir"],
        P["pillar0_external_output_dir_v1"])]
    cands += sorted(glob.glob(os.path.join(P["drive_root"], "*", "matched_table.parquet")))
    for c in cands:
        if os.path.exists(c):
            print(f"  labels from {c}")
            return pd.read_parquet(c)
    raise FileNotFoundError("matched_table.parquet not found. Run nb5 cell 7 once, or set "
                            "INDIA_LABEL_SOURCE to its path.")


def _india_hemorrhage_cols(cols):
    """Intracranial only. Scalp and orbital collections are NOT intracranial hemorrhage and
    counting them was worth 63% of the positives in an earlier run of the India pipeline."""
    named = {"b_ich", "b_sdh", "b_sah", "b_edh", "b_ivh", "b_hemorrhagic_contusion"}
    return sorted(c for c in cols if c in named)


_lab = _india_label_table()
_find = [c for c in _lab.columns if c.lower().startswith(("b_", "f_"))]
_hem = _india_hemorrhage_cols(_lab.columns)
if not _hem:
    raise ValueError("no India hemorrhage column matched; check the b_ column names")
_neg = ("no_", "_no_", "normal", "absent", "unremarkable")
_signal = [c for c in _find if not any(t in c.lower() for t in _neg)]

_per_study = _lab.groupby("StudyInstanceUID", as_index=False)[
    list(dict.fromkeys(_hem + _signal))].max()
_per_study["hemorrhage"] = (_per_study[_hem].to_numpy() > 0).any(axis=1).astype(int)
_per_study["abnormal_any"] = (_per_study[_signal].to_numpy() > 0).any(axis=1).astype(int)

for _b in BACKBONES_TO_RUN:
    _p = os.path.join(INDIA_DIR, f"features_{_b}.parquet")
    if not os.path.exists(_p):
        print(f"  [skip] {_b}: no India feature table at {_p}")
        continue
    _t = pd.read_parquet(_p)
    _t = _t.rename(columns={"series_uid": "series", "patient_id": "patient",
                            "study_uid": "study"})
    _fc = sorted(c for c in _t.columns if c.startswith("emb_"))
    if not _fc:
        _fc = sorted(c for c in _t.columns if c.startswith("f_"))
    _t = _t.merge(_per_study[["StudyInstanceUID", "hemorrhage", "abnormal_any"]],
                  left_on="study", right_on="StudyInstanceUID", how="left")
    _n_before = len(_t)
    _t = _t[_t["hemorrhage"].notna()].reset_index(drop=True)
    if len(_t) < _n_before:
        print(f"  [{_b}] dropped {_n_before - len(_t):,} series with no matching report row "
              f"(zero-filling them would manufacture negatives)")

    if IND_META is None:
        IND_META = _t[["series", "patient", "study", "hemorrhage", "abnormal_any"]].copy()
        G_IND = IND_META["patient"].astype(str).to_numpy()
        for _tg in ("hemorrhage", "abnormal_any"):
            Y_IND[_tg] = IND_META[_tg].astype(int).to_numpy()
    else:
        # Every backbone must describe the SAME series in the SAME order, or a curve drawn
        # on one backbone's rows is scored against another's labels.
        if not np.array_equal(IND_META["series"].to_numpy(), _t["series"].to_numpy()):
            _t = _t.set_index("series").loc[IND_META["series"]].reset_index()
    X_IND[_b] = _t[_fc].to_numpy(dtype=np.float32)

    _sd = os.path.join(INDIA_DIR, "spatial_maps", _b)
    if os.path.isdir(_sd):
        _arrs, _miss = [], 0
        for _s in IND_META["series"]:
            _f = os.path.join(_sd, f"{_s}.npy")
            if os.path.exists(_f):
                _arrs.append(np.load(_f))
            else:
                _miss += 1
                _arrs = None
                break
        if _arrs is not None:
            M_IND[_b] = np.stack(_arrs).astype(np.float32)
    print(f"  [{_b}] India {X_IND[_b].shape[0]:,} x {X_IND[_b].shape[1]:,}"
          + (f"  maps {M_IND[_b].shape[1:]}" if _b in M_IND else "  (no spatial maps)"))

if not X_IND:
    raise FileNotFoundError(f"no India feature tables under {INDIA_DIR}")

print()
print(f"India cohort: {len(IND_META):,} series / {len(set(G_IND)):,} patients")
for _tg, _y in Y_IND.items():
    print(f"  {_tg:14s} {int(_y.sum()):,} positive ({100.0 * _y.mean():.1f}%)")
print(f"hemorrhage columns: {_hem}")


In [ ]:
# ==============================================================================================
# OV-3. THE PRIMITIVES the three experiments share
# ==============================================================================================
# Everything here is built from nb2's own Reducer / AdapterModel / train_adapter, so an arm
# fitted here is the same object the Phase-B grid fits. Nothing is reimplemented.
import copy

import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold

OV_CFG = dict(CFG) if isinstance(globals().get("CFG"), dict) else {}
OV_CFG.setdefault("n_qubits", N_QUBITS)
OV_CFG.setdefault("depth", globals().get("DEPTH", 3))
OV_CFG.setdefault("diff_method", DIFF_METHOD)
OV_CFG.setdefault("init_std", INIT_STD)


def fixed_eval_split(y, groups, frac=0.25, seed=12345):
    """ONE evaluation set, shared by every budget, draw and arm.

    If each point on a learning curve got its own test set, the curve would mix label-budget
    effects with test-set sampling, and the two are the same order of magnitude at these
    prevalences. Patient-grouped and label-stratified.
    """
    y = np.asarray(y).astype(int)
    n_splits = max(2, int(round(1.0 / max(frac, 1e-6))))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pool_idx, eval_idx = next(iter(sgkf.split(np.zeros(len(y)), y, groups)))
    return np.asarray(pool_idx), np.asarray(eval_idx)


def draw_budget(y_pool, g_pool, k, seed):
    """k labelled series drawn WHOLE PATIENTS at a time, keeping both classes present.

    Sampling series independently would split a patient across the labelled and unlabelled
    sets, which is not what "we labelled k scans at the new hospital" means and which leaks
    patient identity into the budget.
    """
    rng = np.random.default_rng(seed)
    y_pool = np.asarray(y_pool).astype(int)
    g_pool = np.asarray(g_pool)
    if k >= len(y_pool):
        return np.arange(len(y_pool))
    pats = rng.permutation(np.unique(g_pool))
    idx_by_pat = {p: np.flatnonzero(g_pool == p) for p in pats}
    take, chosen = [], 0
    for p in pats:
        ii = idx_by_pat[p]
        if chosen + len(ii) > k and chosen > 0:
            continue
        take.append(ii)
        chosen += len(ii)
        if chosen >= k:
            break
    sel = np.concatenate(take) if take else np.arange(min(k, len(y_pool)))
    if len(np.unique(y_pool[sel])) < 2:
        # A single-class budget cannot train anything. Swap in the nearest patient carrying
        # the missing class rather than returning a draw that silently produces NaN.
        missing = 1 - int(y_pool[sel][0])
        cand = [p for p in pats if (y_pool[idx_by_pat[p]] == missing).any()]
        if cand:
            sel = np.concatenate([sel, idx_by_pat[cand[0]]])
    return np.unique(sel)


def make_pooled_arm(arm, d_in, seed, encoder_norm=True, family=None, reducer_mode=None):
    """A pooled (Q1-style) adapter, or None for the linear probe."""
    if arm == "linear_probe":
        return None
    family = family or CURVE_FAMILY
    reducer_mode = reducer_mode or CURVE_REDUCER
    red = Reducer(reducer_mode, d_in, OV_CFG["n_qubits"], seed=seed)
    if hasattr(red, "set_encoder_norm"):
        red.set_encoder_norm(encoder_norm)
    probe = AdapterModel("quantum", red, OV_CFG["n_qubits"], OV_CFG["depth"], family=family,
                         diff_method=OV_CFG["diff_method"], init_std=OV_CFG["init_std"],
                         seed=seed)
    tail = probe.tail_params()
    if arm == "quantum":
        return probe
    hidden, _e, _p = match_hidden_width(OV_CFG["n_qubits"], tail)
    return AdapterModel("classical", red, OV_CFG["n_qubits"], OV_CFG["depth"], family=family,
                        hidden=hidden, diff_method=OV_CFG["diff_method"],
                        init_std=OV_CFG["init_std"], seed=seed)


def make_spatial_arm(arm, c_in, seed, family="Q3_quanv"):
    """Q3's deep-placement arm, parameter-matched the way the Phase-B grid matches it."""
    if arm == "linear_probe":
        return None
    probe = SpatialAdapter("quantum", c_in, OV_CFG["n_qubits"], OV_CFG["depth"],
                           family=family, diff_method=OV_CFG["diff_method"],
                           init_std=OV_CFG["init_std"], seed=seed)
    if arm == "quantum":
        return probe
    hidden, _e, _p = match_hidden_width(OV_CFG["n_qubits"], probe.tail_params())
    return SpatialAdapter("classical", c_in, OV_CFG["n_qubits"], OV_CFG["depth"],
                          hidden=hidden, family=family, diff_method=OV_CFG["diff_method"],
                          init_std=OV_CFG["init_std"], seed=seed)


def fit_arm(arm, Xtr, ytr, seed, epochs=None, encoder_norm=True, model=None,
            adapt_mode=None, lr=None, family=None, reducer_mode=None, Mtr=None):
    """Fit one arm and return (model_or_sklearn, fitted_reducer_or_scaler).

    `model` non-None means ADAPT an existing model rather than build a new one, and
    `adapt_mode` says how much of it is unfrozen.
    """
    ytr = np.asarray(ytr).astype(int)
    if arm == "linear_probe":
        sc = StandardScaler().fit(Xtr)
        lrm = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0)
        lrm.fit(sc.transform(Xtr), ytr)
        return lrm, sc

    if model is None:
        if Mtr is not None:
            # The SPATIAL arm. It carries its own conv+BatchNorm projection, so encoder_norm
            # does not apply and there is no Reducer to fit -- the maps go in raw, exactly as
            # the Phase-B grid feeds them.
            model = make_spatial_arm(arm, Mtr.shape[1], seed, family=family)
            for p in model.parameters():
                p.requires_grad_(True)
            train_adapter(model, Mtr, ytr, epochs=epochs or EPOCHS_SPATIAL, lr=lr or LR,
                          batch=BATCH_SPATIAL, beta=ANGLE_L2_BETA,
                          weight_decay=PROJ_WEIGHT_DECAY, alpha=FOCAL_ALPHA,
                          gamma=FOCAL_GAMMA, seed=seed,
                          reducer_prefixes=("reducer.", "proj."))
            return model, None
        # family/reducer_mode were NOT forwarded here in the first version of this notebook,
        # so every E0 row silently used CURVE_FAMILY regardless of the family it was labelled
        # with. That is why Q1/Q2/Q4 came back identical to six decimal places.
        model = make_pooled_arm(arm, Xtr.shape[1], seed, encoder_norm=encoder_norm,
                                family=family, reducer_mode=reducer_mode)
        model.reducer.fit(Xtr, ytr, seed=seed)
        for p in model.parameters():
            p.requires_grad_(True)
    else:
        model = copy.deepcopy(model)
        # Freeze everything, then unfreeze exactly what this adaptation mode allows. The
        # source reducer's StandardScaler is NOT refit -- applying the source normalisation
        # to the target cohort is what "transfer" means here, and refitting it would quietly
        # turn the experiment into per-dataset recentering.
        for p in model.parameters():
            p.requires_grad_(False)
        if adapt_mode == "readout":
            tgt = ["out.", "mlp.2."]
        elif adapt_mode == "tail":
            tgt = ["out.", "mlp.", "q."]
        else:                                   # tail_plus_projection
            tgt = ["out.", "mlp.", "q.", "reducer."]
        for n, p in model.named_parameters():
            if any(n.startswith(t) for t in tgt):
                p.requires_grad_(True)
        if not any(p.requires_grad for p in model.parameters()):
            raise RuntimeError(f"adapt_mode {adapt_mode!r} unfroze nothing")

    Ztr = model.reducer.transform_np(Xtr)
    train_adapter(model, Ztr, ytr, epochs=epochs or EPOCHS, lr=lr or LR, batch=BATCH,
                  beta=ANGLE_L2_BETA, weight_decay=PROJ_WEIGHT_DECAY,
                  alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=seed)
    return model, model.reducer


def score_arm(arm, fitted, helper, Xte, yte):
    if arm == "linear_probe":
        p = fitted.predict_proba(helper.transform(Xte))[:, 1]
    elif helper is None:                      # spatial arm: maps go in raw
        p = predict_proba(fitted, Xte)
    else:
        p = predict_proba(fitted, helper.transform_np(Xte))
    yte = np.asarray(yte).astype(int)
    out = {"auroc": fast_auroc(yte, p), "ap": fast_auprc(yte, p),
           "prevalence": float(yte.mean())}
    try:
        cal = calibration_metrics(yte, p)
        if isinstance(cal, dict):
            out.update({f"cal_{k}": v for k, v in cal.items()})
    except Exception:
        pass
    return out, p


def pretanh_magnitude(model, X):
    """Median |pre-tanh| on X. This is the x-axis of the saturation figure and the number
    that says whether an arm's angles are still moving with the input."""
    if model is None:
        return float("nan")
    if not hasattr(model, "reducer"):
        # Spatial arm: the pre-tanh value is the output of its own proj (conv + BatchNorm).
        model.eval()
        with torch.no_grad():
            z = model.proj(torch.as_tensor(X, dtype=torch.float32))
            return float(torch.median(torch.abs(z)).item())
    model.eval()
    with torch.no_grad():
        Z = torch.as_tensor(model.reducer.transform_np(X), dtype=torch.float32)
        z = model.reducer(Z)
        return float(torch.median(torch.abs(z)).item())


print("overnight primitives ready:",
      "fixed_eval_split, draw_budget, make_pooled_arm, fit_arm, score_arm, pretanh_magnitude")


In [ ]:
# ==============================================================================================
# OV-4. E0 -- SATURATION AUDIT AND ENCODER-NORM PARITY   [resumable]
# ==============================================================================================
# For every pooled family with a TRAINABLE reducer, fit the arm twice -- without and with the
# same non-affine normalisation Q3 already has -- and record both the AUROC and the median
# pre-tanh magnitude. Two columns, one question: are the pooled arms saturated, and does
# fixing it move the number the placement claim rests on?
#
# Read the output like this:
#   pre-tanh magnitude of order 1-3   -> angles respond to the input; the arm is alive
#   pre-tanh magnitude above ~5       -> tanh is already flat; the arm is partly saturated
#   above ~15                         -> effectively constant output, AUROC near 0.5
if "E0" in STAGES:
    _bks = E0_BACKBONES or BACKBONES_TO_RUN
    for _b in _bks:
        if _b not in X_BY_BACKBONE:
            continue
        _X = X_BY_BACKBONE[_b]
        _M = globals().get("M_BY_BACKBONE", {}).get(_b)
        _y = TARGETS[PRIMARY_TARGET]
        _pool, _ev = fixed_eval_split(_y, GROUPS_ALL, frac=HOLDOUT_FRAC, seed=SEED)

        # every (placement, normalisation) condition this backbone will be measured under
        _conds = [("pooled", _fam, _red, _norm)
                  for _fam in E0_FAMILIES for _red in E0_REDUCERS for _norm in E0_NORMS]
        if E0_SPATIAL_FAMILY and _M is not None:
            _conds.append(("spatial", E0_SPATIAL_FAMILY, "-", True))

        for _place, _fam, _red, _norm in _conds:
            for _sd in E0_SEEDS:
                for _arm in ("quantum", "classical"):
                    _key = f"E0|{_b}|{_place}|{_fam}|{_red}|norm{int(_norm)}|s{_sd}|{_arm}"
                    if _key in OV_DONE:
                        continue
                    if out_of_budget(6.0):
                        print(f"  budget spent -- stopping before {_key}")
                        break
                    _t = time.time()
                    try:
                        if _place == "spatial":
                            _fit, _h = fit_arm(_arm, None, _y[_pool], _sd, family=_fam,
                                               Mtr=_M[_pool])
                            _pre1 = pretanh_magnitude(_fit, _M[_pool])
                            _sc, _ = score_arm(_arm, _fit, None, _M[_ev], _y[_ev])
                        else:
                            _fit, _h = fit_arm(_arm, _X[_pool], _y[_pool], _sd,
                                               encoder_norm=_norm, family=_fam,
                                               reducer_mode=_red)
                            _pre1 = pretanh_magnitude(_fit, _X[_pool])
                            _sc, _ = score_arm(_arm, _fit, _h, _X[_ev], _y[_ev])
                        ov_record(_key, stage="E0", backbone=_b, placement=_place,
                                  family=_fam, reducer=_red, encoder_norm=bool(_norm),
                                  seed=int(_sd), arm=_arm, pretanh_after=_pre1,
                                  n_tail=int(_fit.tail_params()),
                                  minutes=(time.time() - _t) / 60.0, **_sc)
                        print(f"  {_b[:12]:12s} {_place:7s} {_fam:12s} norm={int(_norm)} "
                              f"s={_sd:<5} {_arm:9s} AUROC {_sc['auroc']:.3f}  "
                              f"pre-tanh {_pre1:.2f}  ({(time.time() - _t) / 60.0:.1f} min)")
                    except Exception as _e:
                        ov_record(_key, stage="E0", backbone=_b, placement=_place,
                                  family=_fam, reducer=_red, encoder_norm=bool(_norm),
                                  seed=int(_sd), arm=_arm,
                                  error=f"{type(_e).__name__}: {_e}")
                        print(f"  {_key} FAILED: {type(_e).__name__}: {_e}")

    print("\n" + "=" * 94)
    print("E0 SUMMARY -- encoder conditioning, placement, and ansatz, on one set of splits")
    print("=" * 94)
    _all = [r for r in ov_done_keys().values()
            if r.get("stage") == "E0" and r.get("auroc") is not None]
    if not _all:
        print("  nothing recorded yet.")
    else:
        import pandas as pd
        _d = pd.DataFrame(_all)
        print("\nAUROC, mean over seeds (sd in brackets), and median pre-tanh magnitude:")
        _g = (_d.groupby(["backbone", "placement", "family", "encoder_norm", "arm"])
              .agg(auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"),
                   pretanh=("pretanh_after", "median"), n=("auroc", "size")).round(4))
        print(_g.to_string())
        print()
        print("  READ IT LIKE THIS")
        print("  1. pre-tanh above ~5 means tanh is flat and the arm is emitting a near-")
        print("     constant. Compare the encoder_norm=False rows against the True rows.")
        print("  2. the ANSATZ effect is the spread ACROSS family within a fixed")
        print("     (placement, encoder_norm) block. The seed sd tells you how much of that")
        print("     spread is just training noise.")
        print("  3. the PLACEMENT effect is pooled vs spatial at encoder_norm=True. Both are")
        print("     now measured on the same splits with the same seeds, so the difference is")
        print("     finally attributable to placement and nothing else.")
        print()
        print("  The PCA reducers cannot saturate -- no trainable projection, components")
        print("  divided by their own std at fit time -- so Table 1's pca_supervised numbers")
        print("  are unaffected either way. Say that explicitly in the paper.")
else:
    print("E0 skipped (not in STAGES)")


In [ ]:
# ==============================================================================================
# OV-5. E6 -- INDIA LABEL-EFFICIENCY CURVE   [resumable, the powered experiment]
# ==============================================================================================
# Train from scratch on k labelled India series and score on ONE fixed evaluation set.
# This is the published low-data-robustness experiment, run on real clinical data instead of
# MNIST digit pairs. The claim under test is that the quantum arm degrades more gracefully
# than a parameter-matched classical tail as k shrinks.
#
# abnormal_any carries ~859 positives and is the statistically powered target; hemorrhage
# carries ~94 and is the clinically meaningful one. Report both and say which is which.
if "E6" in STAGES:
    for _b in CURVE_BACKBONES:
        if _b not in X_IND:
            print(f"  [skip] {_b}: no India features")
            continue
        _X = X_IND[_b]
        for _tg in CURVE_TARGETS_INDIA:
            if _tg not in Y_IND:
                continue
            _y = Y_IND[_tg]
            _pool, _ev = fixed_eval_split(_y, G_IND, frac=0.25, seed=SEED)
            print(f"\n[E6] {_b} / {_tg}: pool {len(_pool):,} ({int(_y[_pool].sum())} pos) "
                  f"eval {len(_ev):,} ({int(_y[_ev].sum())} pos)")
            for _k in LABEL_BUDGETS:
                for _d in range(N_DRAWS):
                    _sel = draw_budget(_y[_pool], G_IND[_pool], _k, seed=SEED + 1000 * _d + _k)
                    _tr = _pool[_sel]
                    for _arm in CURVE_ARMS:
                        _key = f"E6|{_b}|{_tg}|k{_k}|d{_d}|{_arm}"
                        if _key in OV_DONE:
                            continue
                        if out_of_budget(3.0):
                            print("  budget spent -- stopping cleanly. Re-run to continue.")
                            _k = None
                            break
                        _t = time.time()
                        try:
                            _fit, _h = fit_arm(_arm, _X[_tr], _y[_tr], SEED + _d,
                                               encoder_norm=ENCODER_NORM_DEFAULT)
                            _sc, _ = score_arm(_arm, _fit, _h, _X[_ev], _y[_ev])
                            ov_record(_key, stage="E6", backbone=_b, target=_tg, k=int(_k),
                                      draw=int(_d), arm=_arm, n_train=int(len(_tr)),
                                      n_train_pos=int(_y[_tr].sum()),
                                      minutes=(time.time() - _t) / 60.0, **_sc)
                        except Exception as _e:
                            ov_record(_key, stage="E6", backbone=_b, target=_tg, k=int(_k),
                                      draw=int(_d), arm=_arm,
                                      error=f"{type(_e).__name__}: {_e}")
                    if _k is None:
                        break
                if _k is None:
                    break
                _got = [r for r in ov_done_keys().values()
                        if r.get("stage") == "E6" and r.get("backbone") == _b
                        and r.get("target") == _tg and r.get("k") == _k
                        and r.get("auroc") is not None]
                if _got:
                    import pandas as pd
                    _s = (pd.DataFrame(_got).groupby("arm")["auroc"]
                          .agg(["mean", "std", "count"]))
                    print(f"  k={_k:>4}  " + " | ".join(
                        f"{a} {r['mean']:.3f}+-{(r['std'] if r['count'] > 1 else 0):.3f}"
                        f"(n={int(r['count'])})" for a, r in _s.iterrows()))
else:
    print("E6 skipped (not in STAGES)")


In [ ]:
# ==============================================================================================
# OV-6. E7 -- RSNA -> INDIA FEW-SHOT TRANSFER   [resumable, the headline experiment]
# ==============================================================================================
# Train the adapter on RSNA, then adapt it on k labelled India series and score on the SAME
# fixed India evaluation set E6 uses, so the two curves are directly comparable: E6 is "start
# from nothing", E7 is "start from RSNA". The gap between them is what pretraining on the
# curated Western cohort is actually worth at each label budget, and k=0 is the zero-shot
# cell already measured at AUROC 0.490.
#
# The source reducer's StandardScaler is applied to India UNCHANGED. That is what transfer
# means; refitting it would turn the experiment into per-dataset recentering, which is a
# different experiment (and one Part C already showed recovers little).
if "E7" in STAGES:
    SOURCE_MODELS = {}
    for _b in CURVE_BACKBONES:
        if _b not in X_IND or _b not in X_BY_BACKBONE:
            print(f"  [skip] {_b}: need both cohorts")
            continue
        _Xs, _ys = X_BY_BACKBONE[_b], TARGETS[PRIMARY_TARGET]
        _Xt, _yt = X_IND[_b], Y_IND["hemorrhage"]
        if _Xs.shape[1] != _Xt.shape[1]:
            raise ValueError(f"{_b}: RSNA has {_Xs.shape[1]} features, India "
                             f"{_Xt.shape[1]}. They must come from the same checkpoint.")
        _pool_t, _ev_t = fixed_eval_split(_yt, G_IND, frac=0.25, seed=SEED)
        print(f"\n[E7] {_b}: source RSNA {len(_ys):,} ({int(_ys.sum())} pos) -> "
              f"India pool {len(_pool_t):,} ({int(_yt[_pool_t].sum())} pos) "
              f"eval {len(_ev_t):,} ({int(_yt[_ev_t].sum())} pos)")

        for _arm in CURVE_ARMS:
            _skey = f"E7src|{_b}|{_arm}"
            # Training a source arm on the full RSNA cohort is the longest single unit in
            # this notebook. Do not start one with minutes left on the clock -- it would be
            # killed mid-fit and the whole stage would have nothing to show.
            if out_of_budget(20.0):
                print(f"  not enough budget to train the {_arm} source model "
                      f"({budget_left_min():.0f} min left, needs ~20). Re-run to continue.")
                break
            _t = time.time()
            _src, _h = fit_arm(_arm, _Xs, _ys, SEED, encoder_norm=ENCODER_NORM_DEFAULT)
            SOURCE_MODELS[(_b, _arm)] = (_src, _h)
            _sc0, _ = score_arm(_arm, _src, _h, _Xt[_ev_t], _yt[_ev_t])
            _sc_src, _ = score_arm(_arm, _src, _h, _Xs, _ys)
            ov_record(_skey, stage="E7src", backbone=_b, arm=_arm,
                      india_zeroshot_auroc=_sc0["auroc"], rsna_insample_auroc=_sc_src["auroc"],
                      minutes=(time.time() - _t) / 60.0)
            print(f"  source {_arm:12s} RSNA(in-sample) {_sc_src['auroc']:.3f}  "
                  f"India zero-shot {_sc0['auroc']:.3f}")

        for _k in LABEL_BUDGETS_T:
            _modes = ["zero"] if _k == 0 else (["readout"] + list(ADAPT_MODES))
            for _mode in _modes:
                for _d in range(1 if _k == 0 else N_DRAWS):
                    _sel = (np.array([], dtype=int) if _k == 0 else
                            draw_budget(_yt[_pool_t], G_IND[_pool_t], _k,
                                        seed=SEED + 1000 * _d + _k))
                    _tr = _pool_t[_sel] if _k else None
                    for _arm in CURVE_ARMS:
                        _key = f"E7|{_b}|k{_k}|{_mode}|d{_d}|{_arm}"
                        if _key in OV_DONE:
                            continue
                        if out_of_budget(3.0):
                            print("  budget spent -- stopping cleanly. Re-run to continue.")
                            _k = None
                            break
                        _t = time.time()
                        try:
                            _src, _h = SOURCE_MODELS[(_b, _arm)]
                            if _k == 0:
                                _fit, _hh = _src, _h
                            elif _arm == "linear_probe":
                                # No partial unfreezing exists for a logistic regression, so
                                # the honest comparator is a probe refit on the k local
                                # labels. E6 reports the same thing from scratch.
                                _fit, _hh = fit_arm(_arm, _Xt[_tr], _yt[_tr], SEED + _d)
                            else:
                                _fit, _hh = fit_arm(_arm, _Xt[_tr], _yt[_tr], SEED + _d,
                                                    model=_src, adapt_mode=_mode,
                                                    epochs=max(10, EPOCHS // 4))
                                _hh = _fit.reducer
                            _sc, _ = score_arm(_arm, _fit, _hh, _Xt[_ev_t], _yt[_ev_t])
                            ov_record(_key, stage="E7", backbone=_b, k=int(_k), mode=_mode,
                                      draw=int(_d), arm=_arm,
                                      n_train=int(0 if _tr is None else len(_tr)),
                                      n_train_pos=int(0 if _tr is None else _yt[_tr].sum()),
                                      pretanh=pretanh_magnitude(
                                          _fit if _arm != "linear_probe" else None,
                                          _Xt[_ev_t]),
                                      minutes=(time.time() - _t) / 60.0, **_sc)
                        except Exception as _e:
                            ov_record(_key, stage="E7", backbone=_b, k=int(_k), mode=_mode,
                                      draw=int(_d), arm=_arm,
                                      error=f"{type(_e).__name__}: {_e}")
                    if _k is None:
                        break
                if _k is None:
                    break
            if _k is None:
                break
            _got = [r for r in ov_done_keys().values()
                    if r.get("stage") == "E7" and r.get("backbone") == _b
                    and r.get("k") == _k and r.get("auroc") is not None]
            if _got:
                import pandas as pd
                _s = (pd.DataFrame(_got).groupby("arm")["auroc"].agg(["mean", "count"]))
                print(f"  k={_k:>4}  " + " | ".join(
                    f"{a} {r['mean']:.3f}(n={int(r['count'])})" for a, r in _s.iterrows()))
else:
    print("E7 skipped (not in STAGES)")


In [ ]:
# ==============================================================================================
# OV-7. SUMMARY -- the three tables to read in the morning, written to Drive
# ==============================================================================================
import pandas as pd

_rows = [r for r in ov_done_keys().values() if r.get("auroc") is not None
         or r.get("india_zeroshot_auroc") is not None]
if not _rows:
    print("nothing recorded yet.")
else:
    _df = pd.DataFrame(_rows)
    _out = os.path.join(OUTPUT_DIR, "overnight_summary.csv")
    _df.to_csv(_out, index=False)
    print(f"{len(_df):,} unit(s) -> {_out}\n")

    _e = _df[_df.get("stage") == "E0"]
    if len(_e):
        print("=" * 94); print("E0  encoder saturation"); print("=" * 94)
        print(_e.pivot_table(index=["backbone", "family", "arm"], columns="encoder_norm",
                             values=["auroc", "pretanh_after"]).round(3).to_string())

    for _stage, _title in (("E6", "E6  India label-efficiency (from scratch)"),
                           ("E7", "E7  RSNA -> India few-shot transfer")):
        _s = _df[_df.get("stage") == _stage]
        if not len(_s):
            continue
        print("\n" + "=" * 94); print(_title); print("=" * 94)
        _idx = ["backbone", "target", "k"] if _stage == "E6" else ["backbone", "mode", "k"]
        _idx = [c for c in _idx if c in _s.columns]
        _p = _s.pivot_table(index=_idx, columns="arm", values="auroc",
                            aggfunc=["mean", "count"]).round(3)
        print(_p.to_string())
        print("\n  Read the CURVE, not any single point. The question is whether the quantum")
        print("  column falls away more slowly than the classical one as k shrinks -- that is")
        print("  the published claim, and this is the first time it is being asked of real")
        print("  cross-institutional clinical data.")

    _err = _df[_df.get("error").notna()] if "error" in _df.columns else _df.iloc[:0]
    if len(_err):
        print(f"\n!! {len(_err)} unit(s) failed. First few:")
        for _, _r in _err.head(5).iterrows():
            print(f"   {_r['key']}: {_r['error']}")

print("\nRemaining work: re-run this notebook to continue. Finished units are skipped.")
